# Introduction au traitement du langage avec spaCy

## Téléchargement du modèle

Avant d'utiliser spaCy, il faut télécharger un modèle pré-entraîné : ici `fr_core_news_sm`, le petit modèle français.

In [1]:
!python -m spacy download fr_core_news_sm  # petit modèle français
import spacy

✔ Download and installation successful
You can now load the package via spacy.load('fr_core_news_sm')


## Traitement d'un document

In [2]:
# Création de l'objet qui traite les documents : le pipeline du modèle
nlp = spacy.load("fr_core_news_sm")

In [3]:
# On crée un texte composé de plusieurs phrases
texte = (
  "Barack Obama fut le 44ème président des États-Unis. "
  "Jean mange une pomme acidulée. "
  "une phrase sans majuscule ni verbe ni point"
)

# On passe le texte par le pipeline
doc = nlp(texte)

## Tokenisation

Affichage des tokens

In [4]:
for jeton in doc:
  print(jeton.text)

Barack
Obama
fut
le
44ème
président
des
États-Unis
.
Jean
mange
une
pomme
acidulée
.
une
phrase
sans
majuscule
ni
verbe
ni
point


## Affichage des phrases

In [5]:
# On affiche les phrases
for phrase in doc.sents:
  print("==================")
  print(phrase)

Barack Obama fut le 44ème président des États-Unis.
Jean mange une pomme acidulée.
une phrase sans majuscule ni verbe ni point


Les phrases viennent de l'analyse syntaxique, pas de la seule ponctuation : la troisième, sans majuscule ni point, est bien isolée.

## Référence au texte d'origine et étiquettes morpho-syntaxiques

In [6]:
# On affiche les tokens de chaque phrase, leur position de départ dans le texte
# d'origine, leur étiquette morpho-syntaxique (POS) et leurs traits morphologiques
for phrase in doc.sents:
  print("=============")
  for jeton in phrase:
    print(f"{jeton.text:15} {jeton.idx:<3} {jeton.pos_:5} {jeton.morph}")

Barack          0   PROPN Gender=Masc|Number=Sing
Obama           7   PROPN 
fut             13  AUX   Mood=Ind|Number=Sing|Person=3|Tense=Past|VerbForm=Fin
le              17  DET   Definite=Def|Gender=Masc|Number=Sing|PronType=Art
44ème           20  ADJ   NumType=Ord|Number=Sing
président       26  NOUN  Gender=Masc|Number=Sing
des             36  ADP   Definite=Def|Number=Plur|PronType=Art
États-Unis      40  PROPN 
.               50  PUNCT 
Jean            52  PROPN Gender=Fem|Number=Sing
mange           57  VERB  Mood=Ind|Number=Sing|Person=3|Tense=Pres|VerbForm=Fin
une             63  DET   Definite=Ind|Gender=Fem|Number=Sing|PronType=Art
pomme           67  NOUN  Gender=Fem|Number=Sing
acidulée        73  VERB  Gender=Fem|Number=Sing|Tense=Past|VerbForm=Part
.               81  PUNCT 
une             83  DET   Definite=Ind|Gender=Fem|Number=Sing|PronType=Art
phrase          87  NOUN  Gender=Fem|Number=Sing
sans            94  ADP   
majuscule       99  NOUN  Gender=Masc|Number

L'attribut `morph` donne les traits morphologiques du token : mode, temps et personne de `fut` (`Mood=Ind|Number=Sing|Person=3|Tense=Past|VerbForm=Fin`), genre et nombre des noms, des adjectifs et des déterminants. Le modèle se trompe aussi ici : `Jean` est au féminin (`Gender=Fem`).

## Lemmatisation


In [7]:
for phrase in doc.sents:
  print("=============")
  for jeton in phrase:
    print(f"{jeton.text:15} {jeton.lemma_}")

Barack          Barack
Obama           Obama
fut             être
le              le
44ème           44èm
président       président
des             de
États-Unis      États-Unis
.               .
Jean            Jean
mange           manger
une             un
pomme           pomme
acidulée        aciduler
.               .
une             un
phrase          phrase
sans            sans
majuscule       majuscule
ni              ni
verbe           verb
ni              ni
point           point


Un modèle statistique se trompe : `44ème` devient `44èm`, qui n'est pas un mot ; `acidulée`, adjectif ici, est étiqueté `VERB` et ramené à l'infinitif `aciduler`. Les formes conjuguées, elles, retrouvent leur lemme (`fut` → `être`, `mange` → `manger`).

## Entités nommées

In [8]:
for entite in doc.ents:
  print(f"{entite.text:20} {entite.label_}")

Barack Obama         PER
États-Unis           LOC
Jean                 PER


In [9]:
spacy.displacy.render(doc, style="ent", jupyter=True)

Le modèle trouve deux personnes (`PER` : `Barack Obama`, `Jean`) et un lieu (`LOC` : `États-Unis`). Son jeu d'étiquettes est réduit (`PER`, `LOC`, `ORG`, `MISC`) : `44ème` n'est pas une entité pour lui.

## Dépendances syntaxiques

- `text` : texte original.
- `dep_` : relation syntaxique entre le token et sa tête (le mot dont il dépend) dans l'arbre.
- `head.text` : texte original de la tête.
- `head.pos_` : étiquette morpho-syntaxique de la tête.
- `children` : liste des mots qui dépendent du token.

La racine de chaque phrase (`ROOT`) est sa propre tête : `président` dans la première, et non `fut`, simple copule ; `Barack` a pour tête `président`, cinq mots plus loin.

In [10]:
for jeton in doc:
  print(
    f"{jeton.text:15} {jeton.dep_:10} {jeton.head.text:15} "
    f"{jeton.head.pos_:5} {' '.join(map(str, jeton.children))}"
  )

Barack          nsubj      président       NOUN  Obama
Obama           flat:name  Barack          PROPN 
fut             cop        président       NOUN  
le              det        président       NOUN  
44ème           amod       président       NOUN  
président       ROOT       président       NOUN  Barack fut le 44ème États-Unis .
des             case       États-Unis      PROPN 
États-Unis      nmod       président       NOUN  des
.               punct      président       NOUN  
Jean            nsubj      mange           VERB  
mange           ROOT       mange           VERB  Jean pomme .
une             det        pomme           NOUN  
pomme           obj        mange           VERB  une acidulée
acidulée        amod       pomme           NOUN  
.               punct      mange           VERB  
une             det        phrase          NOUN  
phrase          ROOT       phrase          NOUN  une majuscule point
sans            case       majuscule       NOUN  
majuscule       n

## Un résumé lisible

In [11]:
spacy.displacy.render(doc, style="dep", jupyter=True, options=dict(distance=100))

Il existe d'autres outils de traitement automatique du langage :

- [Transformers](https://huggingface.co/docs/transformers) (Hugging Face), pour les modèles de langue pré-entraînés
- [Stanza](https://stanfordnlp.github.io/stanza/), la bibliothèque Python de Stanford NLP
- [NLTK](https://www.nltk.org/)